# Peel-CQR Mass Table with Calibrated Uncertainties
### Producing the full data table for *Atomic Data and Nuclear Data Tables*

This Colab notebook produces a complete table of nuclear masses from the proton to the neutron drip line, each with **68% and 95% prediction intervals** from Peel-CQR, together with derived quantities ($S_n$, $S_{2n}$, $S_p$, $S_{2p}$, $Q_\alpha$, $Q_{\beta^-}$) and their intervals, a back-test of the stated coverage, and an auto-generated *Explanation of Tables*.

**Outputs** (zipped and downloaded at the end):

| File | Content |
|---|---|
| `mass_table_full.csv` | every column, every nucleus |
| `mass_table.xlsx` | table + explanation + validation + calibration sheets |
| `mass_table_ascii.txt` | fixed-width machine-readable table (AME-style header) |
| `mass_table_longtable.tex` | printable LaTeX longtable for the journal |
| `explanation_of_tables.md` / `.tex` | column definitions and the uncertainty statement, with the numbers filled in |
| `figures/*.pdf` | interval width on the chart, back-test coverage, peeling curves |

**How the uncertainty is defined** (details in Section 9):

* Masses: Peel-CQR conformal intervals, calibrated separately at each extrapolation distance $d=1,\dots,K$ from the measured chart; linear extension beyond $K$ (flagged).
* Derived quantities: intervals obtained from the mass intervals by the union (Bonferroni) bound, hence **conservative**: a 68% (95%) derived interval combines 84% (97.5%) mass intervals.
* Measured nuclei: the AME2020 experimental value is given alongside the model value.

> **Base model.** The quality of the table depends mostly on the physics base model. Upload a WS4, FRDM(2012) or HFB-24 table in Section 3. Without one, a liquid-drop formula is used — fine for testing the pipeline, **not** for publication.

Run the cells in order (`Runtime → Run all`). Runtime on the free Colab CPU: about 10–15 minutes; with a GPU runtime (`Runtime → Change runtime type → T4 GPU`) the quantile models run in XGBoost on the GPU, all quantile levels in one model, and the final fit and the back-test run in parallel.

In [ ]:
# ---------------------------------------------------------------- configuration
ALPHA_LEVELS   = [0.68, 0.95]          # levels reported for masses and derived quantities
K_PEEL         = 3                     # number of peeled frontier layers
MIN_Z, MIN_N   = 8, 8
Z_MAX          = 118                   # heaviest element in the table
MAX_D_IN_TABLE = None                 # e.g. 6: drop predicted nuclei farther than this from the measured chart
DAMP_LENGTH    = 5.0                   # beyond d > K the ML correction decays as exp(-(d-K)/L); None = no damping
RUN_BACKTEST   = True                  # validate coverage by training on AME2012 and testing on later masses
SEED           = 42

# derived quantities use mass intervals at these levels (union bound): 0.68 -> 0.84, 0.95 -> 0.975
MASS_LEVEL_FOR_DERIVED = {lv: round(1 - (1 - lv) / 2, 4) for lv in ALPHA_LEVELS}
ALL_MASS_LEVELS = sorted(set(ALPHA_LEVELS) | set(MASS_LEVEL_FOR_DERIVED.values()))
print("mass interval levels fitted:", ALL_MASS_LEVELS)

In [ ]:
import sys, subprocess
for pkg in ["openpyxl"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import os, glob, io, shutil, time, zipfile, warnings, urllib.request, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import HistGradientBoostingRegressor

warnings.filterwarnings("ignore")
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
DATA_DIR, OUT_DIR = "ame_data", "adndt_output"
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.family": "serif"})
print("Colab:", IN_COLAB)

## Compute backend: GPU and parallel workers

Settings in the next cell control speed; they do **not** change the method.

| Setting | Default | Effect |
|---|---|---|
| `USE_GPU` | `"auto"` | use NVIDIA GPU(s) when present (Colab, Kaggle, a local CUDA card) |
| `N_JOBS` | `-1` | parallel worker processes (`-1` = all CPU cores) |
| `GPU_WORKERS_PER_GPU` | `"auto"` | workers sharing one GPU; `"auto"` ≈ one per 0.8 GB of GPU memory (each holds a CUDA context) |

What runs where:

* **Gradient-boosted quantiles** → XGBoost (`reg:quantileerror`, all quantile levels in one model) on the GPU; scikit-learn `HistGradientBoostingRegressor` on CPU.
* **Gaussian process** → exact GP in PyTorch on the GPU (Adam on the marginal likelihood, Cholesky solves); scikit-learn on CPU.
* **MLP ensemble** → PyTorch on the GPU; scikit-learn on CPU.
* **Quantile regression forest** weights → PyTorch on the GPU.
* **Independent runs** (folds, subsamples, ablations, simulation replications) → spread over worker processes with `joblib`; with several GPUs, workers alternate between them (Kaggle's 2×T4 are both used).

Any GPU step that fails falls back to the CPU implementation and prints one warning. The GPU and CPU learners are different implementations, so numbers can differ slightly between backends (not between parallel and sequential runs on the same backend). The data set is small (~2,500 nuclei), so the largest gains come from parallel workers and from the GP; a single gradient-boosting fit is about as fast on CPU as on GPU.

In [ ]:
# ---------------------------------------------------------------- compute backend
import os, time, warnings, functools
import numpy as np
from joblib import Parallel, delayed

USE_GPU = "auto"            # "auto" | True | False
N_JOBS = -1                 # -1 = all cores
GPU_WORKERS_PER_GPU = "auto"   # "auto": about one worker per 0.8 GB of GPU memory, or an integer

try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
try:
    import xgboost as xgb
    HAS_XGB = int(xgb.__version__.split(".")[0]) >= 2      # multi-quantile objective needs XGBoost >= 2.0
except ImportError:
    HAS_XGB = False


def _count_gpus():
    if HAS_TORCH:
        try:
            return torch.cuda.device_count() if torch.cuda.is_available() else 0
        except Exception:
            return 0
    try:
        import subprocess
        out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=10).stdout
        return out.count("GPU ")
    except Exception:
        return 0


N_GPUS = _count_gpus() if USE_GPU in ("auto", True) else 0
GPU_ON = N_GPUS > 0 and (HAS_TORCH or HAS_XGB)
if USE_GPU is True and not GPU_ON:
    print("USE_GPU=True but no usable GPU (or neither torch nor xgboost>=2 installed): using CPU")
if GPU_ON and GPU_WORKERS_PER_GPU == "auto":
    try:
        mem_gb = min(torch.cuda.get_device_properties(i).total_memory for i in range(N_GPUS)) / 1e9
        GPU_WORKERS_PER_GPU = max(1, int(mem_gb // 0.8))
    except Exception:
        GPU_WORKERS_PER_GPU = 3
elif GPU_WORKERS_PER_GPU == "auto":
    GPU_WORKERS_PER_GPU = 3
N_CPU = os.cpu_count() or 1
N_WORKERS = N_CPU if N_JOBS in (-1, None) else max(1, int(N_JOBS))
DEFAULT_DEVICE = "cuda:0" if GPU_ON else "cpu"
_WARNED = set()


def _warn_once(key, msg):
    if key not in _WARNED:
        _WARNED.add(key); print(f"[backend] {msg} -> falling back to CPU")


def device_for(i):
    return f"cuda:{i % N_GPUS}" if GPU_ON else "cpu"


def pmap(fn, tasks, desc="", n_jobs=None):
    # run fn(**task, device=...) for every task dict; results in task order
    n = len(tasks)
    nj = n_jobs or (min(N_WORKERS, GPU_WORKERS_PER_GPU * N_GPUS) if GPU_ON else N_WORKERS)
    nj = max(1, min(nj, n))
    t0 = time.time()
    if nj == 1:
        res = [fn(**t, device=device_for(i)) for i, t in enumerate(tasks)]
    else:
        res = Parallel(n_jobs=nj, backend="loky")(delayed(fn)(**t, device=device_for(i)) for i, t in enumerate(tasks))
    if desc:
        print(f"{desc}: {n} tasks on {nj} worker(s) in {time.time() - t0:.0f}s")
    return res


# ---- quantile gradient boosting: XGBoost on GPU, scikit-learn on CPU
from sklearn.ensemble import HistGradientBoostingRegressor
GB_KW = dict(max_iter=600, learning_rate=0.04, max_leaf_nodes=15, min_samples_leaf=8, l2_regularization=1.0,
             random_state=42)
XGB_KW = dict(n_estimators=600, learning_rate=0.04, tree_method="hist", grow_policy="lossguide", max_leaves=15,
              max_depth=0, min_child_weight=8, reg_lambda=1.0, random_state=42)


class MultiQuantile:
    # fits all quantile levels `taus`; predict(X) -> (n, len(taus)) sorted along axis 1 (no crossing)
    def __init__(self, taus, device=None):
        self.taus = list(taus); self.device = device or DEFAULT_DEVICE
        self.use_xgb = HAS_XGB and self.device.startswith("cuda")

    def _fit_sklearn(self, X, y):
        self.ms = [HistGradientBoostingRegressor(loss="quantile", quantile=t, **GB_KW).fit(X, y) for t in self.taus]
        self.use_xgb = False

    def fit(self, X, y):
        if self.use_xgb:
            try:
                self.xm = xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=np.array(self.taus),
                                           device=self.device, **XGB_KW).fit(X, y)
                return self
            except Exception as e:
                _warn_once("xgb", f"XGBoost GPU fit failed ({type(e).__name__}: {str(e)[:80]})")
        self._fit_sklearn(X, y)
        return self

    def predict(self, X):
        if self.use_xgb:
            P = np.asarray(self.xm.predict(X)).reshape(len(X), -1)
        else:
            P = np.column_stack([m.predict(X) for m in self.ms])
        return np.sort(P, axis=1)


# ---- Gaussian process: exact GP in torch on GPU, scikit-learn on CPU
def gp_interval(Xtr, ytr, Xte, z, ls0, device=None, iters=200, lr=0.05):
    device = device or DEFAULT_DEVICE
    if HAS_TORCH and device.startswith("cuda"):
        try:
            return _gp_torch(Xtr, ytr, Xte, z, ls0, device, iters, lr)
        except Exception as e:
            _warn_once("gp", f"torch GP failed ({type(e).__name__}: {str(e)[:80]})")
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as Ck
    gp = GaussianProcessRegressor(Ck(1.0) * RBF(list(ls0)) + WhiteKernel(0.1), normalize_y=True, random_state=42).fit(Xtr, ytr)
    m, s = gp.predict(Xte, return_std=True)
    return m - z * s, m + z * s


def _gp_torch(Xtr, ytr, Xte, z, ls0, device, iters, lr):
    dt = torch.float64
    X = torch.as_tensor(Xtr, dtype=dt, device=device); Xs = torch.as_tensor(Xte, dtype=dt, device=device)
    ym, ys = float(np.mean(ytr)), float(np.std(ytr) or 1.0)
    y = torch.as_tensor((np.asarray(ytr) - ym) / ys, dtype=dt, device=device)[:, None]
    log_ls = torch.tensor(np.log(ls0), dtype=dt, device=device, requires_grad=True)
    log_sf = torch.zeros((), dtype=dt, device=device, requires_grad=True)
    log_sn = torch.tensor(np.log(0.3), dtype=dt, device=device, requires_grad=True)
    eye = torch.eye(len(X), dtype=dt, device=device)

    def kern(A, B):
        ls = log_ls.exp()
        return torch.exp(2 * log_sf) * torch.exp(-0.5 * torch.cdist(A / ls, B / ls) ** 2)

    def chol(K):
        for jit in (1e-8, 1e-6, 1e-4):
            L, info = torch.linalg.cholesky_ex(K + jit * eye)
            if int(info) == 0:
                return L
        raise RuntimeError("Cholesky failed")

    opt = torch.optim.Adam([log_ls, log_sf, log_sn], lr=lr)
    for _ in range(iters):
        opt.zero_grad()
        L = chol(kern(X, X) + torch.exp(2 * log_sn) * eye)
        a = torch.cholesky_solve(y, L)
        nll = 0.5 * (y * a).sum() + torch.log(torch.diagonal(L)).sum()
        nll.backward(); opt.step()
    with torch.no_grad():
        L = chol(kern(X, X) + torch.exp(2 * log_sn) * eye)
        a = torch.cholesky_solve(y, L); Ks = kern(Xs, X)
        mu = (Ks @ a)[:, 0]
        v = torch.linalg.solve_triangular(L, Ks.T, upper=False)
        var = torch.exp(2 * log_sf) - (v ** 2).sum(0) + torch.exp(2 * log_sn)
        sd = torch.sqrt(var.clamp_min(1e-12))
    mu = mu.cpu().numpy() * ys + ym; sd = sd.cpu().numpy() * ys
    return mu - z * sd, mu + z * sd


# ---- MLP ensemble: torch on GPU, scikit-learn on CPU
def mlp_ensemble_interval(Xtr, ytr, Xte, z, n_models=5, device=None, seed=42):
    device = device or DEFAULT_DEVICE
    from sklearn.preprocessing import StandardScaler
    sc = StandardScaler().fit(Xtr); A, B = sc.transform(Xtr), sc.transform(Xte)
    ptr = pte = None
    if HAS_TORCH and device.startswith("cuda"):
        try:
            ptr, pte = _mlp_torch(A, np.asarray(ytr), B, n_models, device, seed)
        except Exception as e:
            _warn_once("mlp", f"torch MLP failed ({type(e).__name__}: {str(e)[:80]})")
    if ptr is None:
        from sklearn.neural_network import MLPRegressor
        ptr, pte = [], []
        for r in range(n_models):
            m = MLPRegressor(hidden_layer_sizes=(64, 64), alpha=1e-3, max_iter=3000, early_stopping=True,
                             random_state=seed + r).fit(A, ytr)
            ptr.append(m.predict(A)); pte.append(m.predict(B))
        ptr, pte = np.array(ptr), np.array(pte)
    sig2 = np.mean((np.asarray(ytr) - ptr.mean(0)) ** 2)
    s = np.sqrt(pte.var(0) + sig2); m = pte.mean(0)
    return m - z * s, m + z * s


def _mlp_torch(A, y, B, n_models, device, seed, max_epochs=3000, patience=100):
    dt = torch.float32
    Xa = torch.as_tensor(A, dtype=dt, device=device); Xb = torch.as_tensor(B, dtype=dt, device=device)
    ym, ys = y.mean(), y.std() or 1.0
    yt = torch.as_tensor((y - ym) / ys, dtype=dt, device=device)[:, None]
    ptr, pte = [], []
    for r in range(n_models):
        torch.manual_seed(seed + r)
        perm = np.random.default_rng(seed + r).permutation(len(A)); nv = max(1, len(A) // 10)
        iv, it = torch.as_tensor(perm[:nv], device=device), torch.as_tensor(perm[nv:], device=device)
        net = torch.nn.Sequential(torch.nn.Linear(A.shape[1], 64), torch.nn.ReLU(), torch.nn.Linear(64, 64),
                                  torch.nn.ReLU(), torch.nn.Linear(64, 1)).to(device)
        opt = torch.optim.Adam(net.parameters(), lr=3e-3, weight_decay=1e-4)
        best, best_state, wait = np.inf, None, 0
        for ep in range(max_epochs):
            opt.zero_grad()
            loss = torch.mean((net(Xa[it]) - yt[it]) ** 2); loss.backward(); opt.step()
            with torch.no_grad():
                v = float(torch.mean((net(Xa[iv]) - yt[iv]) ** 2))
            if v < best - 1e-6:
                best, wait = v, 0; best_state = {k: t.clone() for k, t in net.state_dict().items()}
            else:
                wait += 1
                if wait > patience:
                    break
        net.load_state_dict(best_state)
        with torch.no_grad():
            ptr.append(net(Xa)[:, 0].cpu().numpy() * ys + ym); pte.append(net(Xb)[:, 0].cpu().numpy() * ys + ym)
    return np.array(ptr), np.array(pte)


print(f"backend: {N_CPU} CPU core(s), {min(N_WORKERS, GPU_WORKERS_PER_GPU * N_GPUS) if GPU_ON else N_WORKERS} worker(s) "
      f"| GPUs: {N_GPUS} | GPU on: {GPU_ON} "
      f"| torch: {HAS_TORCH} | xgboost>=2: {HAS_XGB} | default device: {DEFAULT_DEVICE}")

In [ ]:
# ---- backend self-check (fast): GPU and CPU learners should give similar quantile coverage
if GPU_ON:
    rs = np.random.default_rng(0); Xc = rs.normal(size=(1500, 5)); yc = Xc[:, 0] + np.sin(2 * Xc[:, 1]) + 0.3 * rs.normal(size=1500)
    for dev in ["cpu", DEFAULT_DEVICE]:
        t0 = time.time(); P = MultiQuantile((0.05, 0.5, 0.95), device=dev).fit(Xc[:1000], yc[:1000]).predict(Xc[1000:])
        cov = np.mean((yc[1000:] >= P[:, 0]) & (yc[1000:] <= P[:, 2]))
        print(f"quantile GB on {dev:7s}: {time.time() - t0:5.2f}s, 90% band coverage {cov:.3f}")
    for dev in ["cpu", DEFAULT_DEVICE]:
        t0 = time.time(); lo, hi = gp_interval(Xc[:1000], yc[:1000], Xc[1000:], 1.645, [1.0] * 5, device=dev)
        print(f"GP on {dev:7s}: {time.time() - t0:5.2f}s, 90% coverage {np.mean((yc[1000:] >= lo) & (yc[1000:] <= hi)):.3f}")
else:
    print("no GPU: everything runs on CPU workers")

## 1. AME mass tables

AME2020 is required. AME2012 is needed only for the back-test (Section 8); without it the table is still produced. The cell looks in `ame_data/`, the Colab working folder and `DRIVE_DIR`, then downloads from this project's GitHub repository (AME2012 and AME2020 are stored there) and from the AMDC mirrors. The IAEA server often refuses requests from Colab (HTTP 403): then an **upload button** appears. Download the files in your own browser from https://www-nds.iaea.org/amdc/ (`mass_1.mas20.txt`, and `mass.mas12` for the back-test) and select them there. Files can also be kept on Google Drive: mount Drive and set `DRIVE_DIR`.

In [ ]:
AME = {
    2012: ("mass.mas12", ["https://raw.githubusercontent.com/yamenetoo/nuclear-mass-peel-cqr/main/data/ame/mass.mas12",
                          "https://www-nds.iaea.org/amdc/ame2012/mass.mas12",
                          "https://amdc.impcas.ac.cn/masstables/Ame2012/mass.mas12",
                          "http://amdc.in2p3.fr/masstables/Ame2012/mass.mas12"]),
    2016: ("mass16.txt", ["https://www-nds.iaea.org/amdc/ame2016/mass16.txt",
                          "https://amdc.impcas.ac.cn/masstables/Ame2016/mass16.txt",
                          "http://amdc.in2p3.fr/masstables/Ame2016/mass16.txt"]),
    2020: ("mass_1.mas20.txt", ["https://raw.githubusercontent.com/yamenetoo/nuclear-mass-peel-cqr/main/data/ame/mass_1.mas20.txt",
                                "https://www-nds.iaea.org/amdc/ame2020/mass_1.mas20.txt",
                                "https://www-nds.iaea.org/amdc/ame2020/mass_1.mas20",
                                "https://amdc.impcas.ac.cn/masstables/Ame2020/mass_1.mas20.txt"]),
}
DRIVE_DIR = None   # e.g. "/content/drive/MyDrive/ame"  (mount Drive first); files there are used directly
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
                         "Chrome/124.0 Safari/537.36", "Referer": "https://www-nds.iaea.org/amdc/"}


def edition_of(name):
    # guess the AME edition from an uploaded file name (mass.mas12, mass16.txt, mass_1.mas20.txt, ...)
    n = os.path.basename(name).lower()
    for y, keys in {2012: ("mas12", "mass12", "ame2012"), 2016: ("mass16", "mas16", "ame2016"),
                    2020: ("mas20", "mass20", "ame2020")}.items():
        if any(k in n for k in keys):
            return y
    return None


def try_get(year, download=True):
    fname, urls = AME[year]
    path = os.path.join(DATA_DIR, fname)
    if os.path.exists(path) and os.path.getsize(path) > 100_000:
        return path
    for root in [r for r in [DATA_DIR, "data/ame", "../data/ame", ".", "/content", DRIVE_DIR] if r and os.path.isdir(r)]:
        for h in glob.glob(os.path.join(root, "**", "*"), recursive=True):
            if os.path.isfile(h) and edition_of(h) == year and os.path.getsize(h) > 100_000 \
                    and os.path.abspath(h) != os.path.abspath(path):
                shutil.copy(h, path); print(f"AME{year}: using {h}"); return path
    if not download:
        return None
    for url in urls:
        try:
            try:
                import requests
                r = requests.get(url, headers=HEADERS, timeout=20); r.raise_for_status(); data = r.content
            except ImportError:
                with urllib.request.urlopen(urllib.request.Request(url, headers=HEADERS), timeout=20) as r:
                    data = r.read()
            if len(data) > 100_000:
                open(path, "wb").write(data); print(f"AME{year}: downloaded from {url}"); return path
        except Exception as e:
            print(f"AME{year}: {url} -> {type(e).__name__}: {str(e)[:80]}")
    return None


paths = {y: try_get(y) for y in AME}
missing = [y for y, p in paths.items() if p is None]
if missing and IN_COLAB:
    print("\nThe server refused the download. In your browser, open https://www-nds.iaea.org/amdc/ and save:")
    for y in missing:
        print(f"   AME{y}: {AME[y][0]}" + ("   (required)" if y == 2020 else "   (optional, for the back-test)"))
    print("then select those files with the button below (Cancel to continue without the optional ones).")
    try:
        up = colab_files.upload()
    except Exception as e:
        up = {}; print("upload skipped:", e)
    for name, content in up.items():
        y = edition_of(name)
        if y is None:
            print(f"  could not tell which edition '{name}' is; rename it to mass.mas12 / mass16.txt / mass_1.mas20.txt")
            continue
        open(os.path.join(DATA_DIR, AME[y][0]), "wb").write(content); print(f"  {name} -> AME{y}")
    paths = {y: try_get(y, download=False) for y in AME}
if paths[2020] is None:
    raise FileNotFoundError("AME2020 (mass_1.mas20.txt) is required: upload it or put it in ame_data/.")
if RUN_BACKTEST and (paths[2012] is None):
    print("AME2012 missing -> back-test disabled"); RUN_BACKTEST = False
print({y: p for y, p in paths.items()})

In [ ]:
DELTA_H, DELTA_N = 7.288971064, 8.071318062      # MeV
SYMBOLS = ("n H He Li Be B C N O F Ne Na Mg Al Si P S Cl Ar K Ca Sc Ti V Cr Mn Fe Co Ni Cu Zn Ga Ge As Se Br Kr "
           "Rb Sr Y Zr Nb Mo Tc Ru Rh Pd Ag Cd In Sn Sb Te I Xe Cs Ba La Ce Pr Nd Pm Sm Eu Gd Tb Dy Ho Er Tm Yb "
           "Lu Hf Ta W Re Os Ir Pt Au Hg Tl Pb Bi Po At Rn Fr Ra Ac Th Pa U Np Pu Am Cm Bk Cf Es Fm Md No Lr Rf "
           "Db Sg Bh Hs Mt Ds Rg Cn Nh Fl Mc Lv Ts Og 119 120").split()


def parse_ame(path):
    rows = []
    for line in open(path, encoding="latin-1"):
        if len(line) < 40:
            continue
        try:
            N = int(line[4:9]); Z = int(line[9:14]); A = int(line[14:19])
        except ValueError:
            continue
        if N + Z != A:
            continue
        t = line[28:].split()
        if len(t) < 2:
            continue
        try:
            me = float(t[0].replace("#", "")) / 1000; dme = float(t[1].replace("#", "")) / 1000
        except ValueError:
            continue
        rows.append((Z, N, A, me, dme, "#" in t[0]))
    d = pd.DataFrame(rows, columns=["Z", "N", "A", "ME", "dME", "estimated"]).drop_duplicates(["Z", "N"])
    d["B"] = d.Z * DELTA_H + d.N * DELTA_N - d.ME
    return d.reset_index(drop=True)


ame = {y: parse_ame(p) for y, p in paths.items() if p}
a20 = ame[2020]
B_ALPHA = float(a20[(a20.Z == 2) & (a20.N == 2)].B.iloc[0]) if ((a20.Z == 2) & (a20.N == 2)).any() else 28.295674
print(f"B(4He) = {B_ALPHA:.6f} MeV")
for y, d in ame.items():
    print(f"AME{y}: measured {(~d.estimated).sum()}, estimated {d.estimated.sum()}")
EXP = a20[(~a20.estimated) & (a20.Z >= MIN_Z) & (a20.N >= MIN_N)].set_index(["Z", "N"])
EXP_ALL = a20[(~a20.estimated) & (a20.Z >= 1)].set_index(["Z", "N"])   # all measured, for neighbours in derived quantities
EST = a20[(a20.estimated) & (a20.Z >= MIN_Z) & (a20.N >= MIN_N)].set_index(["Z", "N"])
print("measured nuclei used for fitting:", len(EXP))

## 2. Base model

Upload a theory table (WS4, FRDM2012, HFB-24…) as a whitespace- or comma-separated text file and set the column numbers below (0-based). Its nuclei also define the candidate grid of the final table. If `BASE_FILE = None`, a liquid-drop formula fitted to the measured masses is used.

In [ ]:
BASE_FILE = None          # e.g. "ws4.txt"   (in Colab: upload it in the Files panel first)
BASE_COLS = dict(z=0, n=1, value=2, skiprows=1, sep=r"\s+")
BASE_KIND = "mass_excess"  # "mass_excess" or "binding" (total binding energy)
BASE_UNIT = "MeV"          # "MeV" or "keV"
BASE_NAME = "LDM" if BASE_FILE is None else os.path.basename(BASE_FILE)


def ldm_design(Z, N):
    Z = np.asarray(Z, float); N = np.asarray(N, float); A = Z + N
    dl = np.where((Z % 2 == 0) & (N % 2 == 0), 1.0, np.where((Z % 2 == 1) & (N % 2 == 1), -1.0, 0.0))
    return np.column_stack([A, -A ** (2 / 3), -Z * (Z - 1) / A ** (1 / 3), -(N - Z) ** 2 / A,
                            (N - Z) ** 2 / A ** (4 / 3), dl * A ** -0.5])


def make_base(train_zn, train_B):
    # returns a function (Z, N) -> B_theory (MeV)
    if BASE_FILE is not None:
        t = pd.read_csv(BASE_FILE, sep=BASE_COLS["sep"], header=None, skiprows=BASE_COLS["skiprows"],
                        comment="#", engine="python")
        v = t.iloc[:, BASE_COLS["value"]].astype(float) / (1000.0 if BASE_UNIT == "keV" else 1.0)
        z = t.iloc[:, BASE_COLS["z"]].astype(int); n = t.iloc[:, BASE_COLS["n"]].astype(int)
        b = (z * DELTA_H + n * DELTA_N - v) if BASE_KIND == "mass_excess" else v
        return BaseModel(table=dict(zip(zip(z, n), b)))
    coef, *_ = np.linalg.lstsq(ldm_design(train_zn[:, 0], train_zn[:, 1]), train_B, rcond=None)
    return BaseModel(coef=coef)


class BaseModel:
    # picklable base model: tabulated (Z,N)->B or liquid-drop coefficients
    def __init__(self, table=None, coef=None):
        self.table, self.coef = table, coef

    def __call__(self, Z, N):
        if self.table is not None:
            return np.array([self.table.get((int(a), int(c)), np.nan) for a, c in zip(Z, N)])
        return ldm_design(Z, N) @ self.coef


if BASE_FILE is None:
    print("WARNING: liquid-drop base. Upload WS4/FRDM2012/HFB and set BASE_FILE before producing the published table.")

## 3. Features, distances, multi-level quantile model

In [ ]:
MAGIC = np.array([2, 8, 20, 28, 50, 82, 126, 184])


def valence(x):
    x = np.asarray(x)[:, None]
    lo = np.where(MAGIC[None] <= x, MAGIC[None], -1).max(1); hi = np.where(MAGIC[None] > x, MAGIC[None], 10**6).min(1)
    xv = x[:, 0]
    return np.minimum(xv - lo, hi - xv), (xv - lo) / (hi - lo)


def features(Z, N):
    Z = np.asarray(Z, float); N = np.asarray(N, float); A = Z + N
    vz, fz = valence(Z.astype(int)); vn, fn = valence(N.astype(int))
    P = np.where(vz + vn > 0, vz * vn / np.maximum(vz + vn, 1), 0.0)
    return np.column_stack([Z, N, A, (N - Z) / A, Z % 2, N % 2, vz, vn, fz, fn, P])


def manhattan_to(ref, pts):
    ref = np.asarray(ref); pts = np.asarray(pts); out = np.empty(len(pts), int)
    for i in range(0, len(pts), 512):
        p = pts[i:i + 512]
        out[i:i + 512] = (np.abs(p[:, None, 0] - ref[None, :, 0]) + np.abs(p[:, None, 1] - ref[None, :, 1])).min(1)
    return out


def depth_in(zn):
    zn = np.asarray(zn); zmax, nmax = zn[:, 0].max() + K_PEEL + 2, zn[:, 1].max() + K_PEEL + 2
    g = np.zeros((zmax + 1, nmax + 1), bool); g[zn[:, 0], zn[:, 1]] = True
    holes = np.argwhere(~g); holes = holes[(holes[:, 0] >= MIN_Z - 1) & (holes[:, 1] >= MIN_N - 1)]
    return manhattan_to(holes, zn)


GB_KW = dict(max_iter=600, learning_rate=0.04, max_leaf_nodes=15, min_samples_leaf=8,
             l2_regularization=1.0, random_state=SEED)


class MultiQuantileGB:
    # median + (lo, hi) quantiles for every level in ONE multi-quantile model (XGBoost on GPU, scikit-learn on CPU);
    # predictions are sorted across levels, so the intervals are nested and never cross
    def __init__(self, levels, device=None):
        self.levels = sorted(levels)
        self.taus = sorted({0.5} | {(1 - lv) / 2 for lv in self.levels} | {1 - (1 - lv) / 2 for lv in self.levels})
        self.mq = MultiQuantile(self.taus, device)

    def fit(self, X, y):
        self.mq.fit(X, y); return self

    def predict(self, X):
        P = self.mq.predict(X); col = {round(t, 6): j for j, t in enumerate(self.taus)}
        med = P[:, col[0.5]]
        out = {lv: (P[:, col[round((1 - lv) / 2, 6)]], P[:, col[round(1 - (1 - lv) / 2, 6)]]) for lv in self.levels}
        return med, out


def conformal_q(s, level):
    s = np.sort(np.asarray(s)); n = len(s)
    if n == 0:
        return np.nan
    k = int(np.ceil((n + 1) * level))
    return s[min(k, n) - 1]

In [ ]:
class PeelCQRPredictor:
    # fit on measured nuclei (zn, B); predict mass intervals for any (Z, N)
    def __init__(self, levels, K=K_PEEL, device=None):
        self.levels, self.K, self.device = sorted(levels), K, device

    def fit(self, zn, B):
        self.zn = np.asarray(zn); self.base = make_base(self.zn, B)
        y = B - self.base(self.zn[:, 0], self.zn[:, 1]); ok = np.isfinite(y)
        self.zn, y = self.zn[ok], y[ok]
        X = features(self.zn[:, 0], self.zn[:, 1])
        dep = depth_in(self.zn); core = dep > self.K
        q_core = MultiQuantileGB(self.levels, self.device).fit(X[core], y[core])
        d_core = manhattan_to(self.zn[core], self.zn[~core])
        _, ints = q_core.predict(X[~core])
        self.c = {}; self.slope = {}; self.n_d = [int(np.sum(d_core == k)) for k in range(1, self.K + 1)]
        for lv in self.levels:
            lo, hi = ints[lv]; s = np.maximum(lo - y[~core], y[~core] - hi)
            cd = np.array([conformal_q(s[d_core == k], lv) for k in range(1, self.K + 1)])
            cd = np.maximum.accumulate(pd.Series(cd).ffill().bfill().values)
            self.c[lv] = cd
            self.slope[lv] = max(0.0, np.polyfit(np.arange(1, self.K + 1), cd, 1)[0]) if self.K >= 2 else 0.0
        for a, b in zip(self.levels[:-1], self.levels[1:]):      # wider level -> constants at least as large
            self.c[b] = np.maximum(self.c[b], self.c[a]); self.slope[b] = max(self.slope[b], self.slope[a])
        self.q = MultiQuantileGB(self.levels, self.device).fit(X, y)
        self.n_core, self.n_peel = int(core.sum()), int((~core).sum())
        return self

    def c_at(self, lv, d):
        d = np.maximum(np.asarray(d), 1)
        inside = self.c[lv][np.minimum(d, self.K) - 1]
        return np.where(d <= self.K, inside, self.c[lv][-1] + self.slope[lv] * (d - self.K))

    def predict(self, Z, N):
        Z = np.asarray(Z); N = np.asarray(N)
        d = manhattan_to(self.zn, np.column_stack([Z, N]))
        base = self.base(Z, N)
        med, ints = self.q.predict(features(Z, N))
        shift = np.zeros_like(med)
        if DAMP_LENGTH:
            w = np.where(d > self.K, np.exp(-(d - self.K) / DAMP_LENGTH), 1.0)
            shift = (1 - w) * med                       # pull the ML correction towards zero far out
        out = {"d": d, "base": base, "B": base + med - shift}
        for lv in self.levels:
            lo, hi = ints[lv]; c = self.c_at(lv, d)
            out[f"lo{lv}"] = base + lo - shift - c
            out[f"hi{lv}"] = base + hi - shift + c
        return out

## 4. Derived quantities and their intervals

For quantities that are differences of two binding energies (plus constants),

$$S_n = B(Z,N)-B(Z,N-1),\quad S_{2n} = B(Z,N)-B(Z,N-2),\quad S_p = B(Z,N)-B(Z-1,N),\quad S_{2p}=B(Z,N)-B(Z-2,N),$$
$$Q_\alpha = B(Z-2,N-2) + B(^4\mathrm{He}) - B(Z,N),\qquad Q_{\beta^-} = (\Delta_n-\Delta_{^1\mathrm{H}}) + B(Z+1,N-1) - B(Z,N),$$

the interval of $X - Y$ is $[\,\mathrm{lo}_X-\mathrm{hi}_Y,\ \mathrm{hi}_X-\mathrm{lo}_Y\,]$. If each mass interval has coverage $\ge 1-\alpha/2$, the union bound gives coverage $\ge 1-\alpha$ for the difference — whatever the correlation between the two errors. Measured masses enter with their AME2020 value (experimental uncertainties are negligible on this scale).

In [ ]:
DERIVED = {  # name: (sign of B(Z,N), neighbour offset (dZ, dN), constant)
    "Sn":   (+1, (0, -1), 0.0),
    "S2n":  (+1, (0, -2), 0.0),
    "Sp":   (+1, (-1, 0), 0.0),
    "S2p":  (+1, (-2, 0), 0.0),
    "Qa":   (-1, (-2, -2), None),     # constant B(4He) filled below
    "Qbm":  (-1, (+1, -1), DELTA_N - DELTA_H),
}


def derived_table(tab, levels_map):
    # tab: DataFrame indexed by (Z,N) with columns B, lo<lv>, hi<lv> (for measured nuclei lo=hi=B)
    out = {}
    idx = tab.index
    for name, (sgn, (dz, dn), const) in DERIVED.items():
        const = B_ALPHA if name == "Qa" else const
        nb = pd.MultiIndex.from_arrays([idx.get_level_values(0) + dz, idx.get_level_values(1) + dn])
        has = nb.isin(tab.index)
        B0 = tab["B"].values
        Bn = np.full(len(tab), np.nan); Bn[has] = tab.loc[nb[has], "B"].values
        # X = sgn*(B0 - Bn) + const
        out[name] = sgn * (B0 - Bn) + const
        for lv, mlv in levels_map.items():
            lo0, hi0 = tab[f"lo{mlv}"].values, tab[f"hi{mlv}"].values
            lon = np.full(len(tab), np.nan); hin = np.full(len(tab), np.nan)
            lon[has] = tab.loc[nb[has], f"lo{mlv}"].values; hin[has] = tab.loc[nb[has], f"hi{mlv}"].values
            a, b = lo0 - hin, hi0 - lon                    # interval of B0 - Bn
            if sgn > 0:
                out[f"{name}_lo{lv}"], out[f"{name}_hi{lv}"] = a + const, b + const
            else:
                out[f"{name}_lo{lv}"], out[f"{name}_hi{lv}"] = -b + const, -a + const
    return pd.DataFrame(out, index=idx)

## 5. Fit the final model on all nuclei measured in AME2020

In [ ]:
zn_meas = np.array(list(EXP.index)); B_meas = EXP.B.values
BT_TRAIN = BT_TEST = None
if RUN_BACKTEST:                                   # the back-test model (Section 8) is fitted now, in parallel
    a12 = ame[2012]
    M12 = set(zip(a12[(~a12.estimated) & (a12.Z >= MIN_Z) & (a12.N >= MIN_N)].Z,
                  a12[(~a12.estimated) & (a12.Z >= MIN_Z) & (a12.N >= MIN_N)].N))
    BT_TRAIN = [k for k in EXP.index if k in M12]
    BT_TEST = [k for k in EXP.index if k not in M12]


def fit_task(zn, B, device=None):
    return PeelCQRPredictor(ALL_MASS_LEVELS, device=device).fit(zn, B)


t0 = time.time()
tasks = [dict(zn=zn_meas, B=B_meas)]
if RUN_BACKTEST:
    tasks.append(dict(zn=np.array(BT_TRAIN), B=EXP.loc[BT_TRAIN].B.values))
fitted = pmap(fit_task, tasks, desc="model fits")
final = fitted[0]; bt = fitted[1] if RUN_BACKTEST else None
print(f"fitted in {time.time() - t0:.0f}s | core {final.n_core}, peeled {final.n_peel}, nuclei per distance {final.n_d}")
calib = pd.DataFrame({f"c_{lv}": final.c[lv] for lv in final.levels}, index=pd.Index(range(1, K_PEEL + 1), name="d"))
calib.loc["slope (MeV/step)"] = [final.slope[lv] for lv in final.levels]
calib.round(3)

## 6. Candidate grid and drip lines

Candidates: every $(Z,N)$ with $8\le Z\le Z_{\max}$ and $N\ge 8$ in a generous band around stability (or every nucleus of the uploaded base table). A nucleus enters the table if it is measured, or if its median $S_n$, $S_{2n}$, $S_p$ and $S_{2p}$ are all positive, walking outward from the most bound isotope of each element until two consecutive nuclei fail. A nucleus is labelled **bound with confidence** when the lower 95% limits of all four separation energies are positive, and **uncertain** otherwise.

In [ ]:
cand = []
for Z in range(MIN_Z - 2, Z_MAX + 1):            # Z-2 / N-2 neighbours are needed for derived quantities
    for N in range(max(MIN_N - 2, 0), int(2.0 * Z + 60)):
        cand.append((Z, N))
cand = np.array(cand)
pred = final.predict(cand[:, 0], cand[:, 1])
grid = pd.DataFrame({"Z": cand[:, 0], "N": cand[:, 1], **pred}).set_index(["Z", "N"])
grid = grid[np.isfinite(grid.B) & (grid.B > 0)]
# measured nuclei: replace by experiment with zero-width intervals (for derived quantities only)
gm = grid.copy()
meas_idx = gm.index.intersection(EXP_ALL.index)
for lv in final.levels:
    gm.loc[meas_idx, f"lo{lv}"] = EXP_ALL.loc[meas_idx, "B"].values
    gm.loc[meas_idx, f"hi{lv}"] = EXP_ALL.loc[meas_idx, "B"].values
gm.loc[meas_idx, "B"] = EXP_ALL.loc[meas_idx, "B"].values
der = derived_table(gm, MASS_LEVEL_FOR_DERIVED)
print("candidate nuclei:", len(grid))

In [ ]:
SEP = ["Sn", "S2n", "Sp", "S2p"]
bound_med = (der[SEP] > 0).all(axis=1)
measured = pd.Series(grid.index.isin(EXP.index), index=grid.index)
keep = set()
for Z in range(MIN_Z, Z_MAX + 1):
    rows = grid.loc[Z] if Z in grid.index.get_level_values(0) else None
    if rows is None:
        continue
    Ns = rows.index.values
    ba = (grid.loc[Z].B / (Z + Ns)).values
    start = Ns[np.argmax(ba)]
    for step in (+1, -1):
        fails, N = 0, start
        while (Z, N) in grid.index and fails < 2:
            ok = measured.get((Z, N), False) or bound_med.get((Z, N), False)
            if ok and N >= MIN_N:
                keep.add((Z, N)); fails = 0
            else:
                fails += 1
            N += step
keep = sorted(keep)
if MAX_D_IN_TABLE is not None:
    keep = [k for k in keep if measured.get(k, False) or grid.loc[k, "d"] <= MAX_D_IN_TABLE]
lo95 = {s: der[f"{s}_lo0.95"] for s in SEP}
conf_bound = pd.concat([lo95[s] > 0 for s in SEP], axis=1).all(axis=1)
print(f"nuclei in table: {len(keep)} | measured {sum(measured.get(k, False) for k in keep)} | "
      f"predicted {sum(not measured.get(k, False) for k in keep)}")

## 7. Assemble the table

In [ ]:
T = grid.loc[keep].copy()
T["A"] = T.index.get_level_values(0) + T.index.get_level_values(1)
T["El"] = [SYMBOLS[z] if z < len(SYMBOLS) else str(z) for z in T.index.get_level_values(0)]
T["status"] = np.where(T.index.isin(EXP.index), "exp", "pred")
T["B_exp"] = EXP.B.reindex(T.index).values
T["dB_exp"] = EXP.dME.reindex(T.index).values
T["ME_exp"] = EXP.ME.reindex(T.index).values
T["B_AME_syst"] = EST.B.reindex(T.index).values            # AME2020 '#' estimates, for comparison only
Zs, Ns = T.index.get_level_values(0).values, T.index.get_level_values(1).values
T["ME"] = Zs * DELTA_H + Ns * DELTA_N - T.B                 # model mass excess (MeV)
for lv in ALPHA_LEVELS:
    T[f"ME_lo{lv}"] = Zs * DELTA_H + Ns * DELTA_N - T[f"hi{lv}"]
    T[f"ME_hi{lv}"] = Zs * DELTA_H + Ns * DELTA_N - T[f"lo{lv}"]
T["beyond_calibration"] = T.d > K_PEEL
T["beyond_training_Z"] = Zs > zn_meas[:, 0].max()
D = der.loc[keep]
T = T.join(D[[c for c in D.columns if any(c.startswith(s) for s in DERIVED)]])
T["bound_95"] = conf_bound.reindex(T.index).fillna(False).values
cols = (["A", "El", "status", "d", "beyond_calibration", "beyond_training_Z", "bound_95",
         "B_exp", "dB_exp", "ME_exp", "B"] + [f"{p}{lv}" for lv in ALPHA_LEVELS for p in ("lo", "hi")]
        + ["ME"] + [f"ME_{p}{lv}" for lv in ALPHA_LEVELS for p in ("lo", "hi")] + ["B_AME_syst"]
        + [c for s in DERIVED for c in [s] + [f"{s}_{p}{lv}" for lv in ALPHA_LEVELS for p in ("lo", "hi")]])
T = T[cols].reset_index()
T = T.rename(columns={**{f"lo{lv}": f"B_lo{lv}" for lv in ALPHA_LEVELS}, **{f"hi{lv}": f"B_hi{lv}" for lv in ALPHA_LEVELS}})
print(T.shape)
T.head()

## 8. Back-test: does the stated coverage hold?

Train the whole pipeline on nuclei measured in **AME2012** only, then score its intervals on the nuclei first measured in AME2016 or AME2020 (true future data, AME2020 values as truth). Masses and the derived quantities are checked at every reported level, with Clopper–Pearson 95% intervals.

In [ ]:
def cp(k, n, a=0.05):
    if n == 0:
        return (np.nan, np.nan)
    return (0.0 if k == 0 else stats.beta.ppf(a / 2, k, n - k + 1), 1.0 if k == n else stats.beta.ppf(1 - a / 2, k + 1, n - k))


back_rows, back_dist = [], []
if RUN_BACKTEST:
    tr, te = BT_TRAIN, BT_TEST                         # model `bt` was fitted in Section 5
    zte = np.array(te); p = bt.predict(zte[:, 0], zte[:, 1]); yb = EXP.loc[te].B.values
    for lv in ALPHA_LEVELS:
        cov = (yb >= p[f"lo{lv}"]) & (yb <= p[f"hi{lv}"]); k, n = int(cov.sum()), len(cov)
        back_rows.append(dict(quantity="B (mass)", level=lv, n=n, coverage=k / n, CP_lo=cp(k, n)[0], CP_hi=cp(k, n)[1],
                              mean_width_MeV=np.mean(p[f"hi{lv}"] - p[f"lo{lv}"])))
        for dd in sorted(set(np.minimum(p["d"], 4))):
            m = np.minimum(p["d"], 4) == dd; kk = int(cov[m].sum())
            back_dist.append(dict(level=lv, d=("4+" if dd == 4 else str(dd)), n=int(m.sum()), coverage=kk / m.sum(),
                                  CP_lo=cp(kk, m.sum())[0], CP_hi=cp(kk, m.sum())[1]))
    # derived quantities on test nuclei: neighbours measured in AME2012 enter with their values
    gb = pd.DataFrame({"Z": zte[:, 0], "N": zte[:, 1], **p}).set_index(["Z", "N"])
    known = EXP.loc[tr, ["B"]].copy(); known["d"] = 0; known["base"] = np.nan
    for lv in bt.levels:
        known[f"lo{lv}"] = known.B; known[f"hi{lv}"] = known.B
    gb = pd.concat([gb, known[gb.columns]]).sort_index()
    dbt = derived_table(gb, MASS_LEVEL_FOR_DERIVED).loc[te]
    truth_tab = EXP_ALL[["B"]].copy()
    for lv in bt.levels:
        truth_tab[f"lo{lv}"] = truth_tab.B; truth_tab[f"hi{lv}"] = truth_tab.B
    dtrue = derived_table(truth_tab, MASS_LEVEL_FOR_DERIVED).loc[te]
    for s in DERIVED:
        ok = np.isfinite(dtrue[s].values) & np.isfinite(dbt[f"{s}_lo{ALPHA_LEVELS[0]}"].values)
        for lv in ALPHA_LEVELS:
            c = (dtrue[s].values[ok] >= dbt[f"{s}_lo{lv}"].values[ok]) & (dtrue[s].values[ok] <= dbt[f"{s}_hi{lv}"].values[ok])
            k, n = int(c.sum()), int(ok.sum())
            if n:
                back_rows.append(dict(quantity=s, level=lv, n=n, coverage=k / n, CP_lo=cp(k, n)[0], CP_hi=cp(k, n)[1],
                                      mean_width_MeV=np.mean(dbt[f"{s}_hi{lv}"].values[ok] - dbt[f"{s}_lo{lv}"].values[ok])))
backtest = pd.DataFrame(back_rows); backtest_d = pd.DataFrame(back_dist)
print(f"back-test: train AME2012 ({len(tr) if RUN_BACKTEST else 0}), test later masses ({len(te) if RUN_BACKTEST else 0})")
backtest.round(3)

In [ ]:
backtest_d.round(3) if len(backtest_d) else "back-test disabled"

## 9. Figures

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
pr = T[T.status == "pred"]; ex = T[T.status == "exp"]
ax.scatter(ex.N, ex.Z, s=3, c="#adb5bd", marker="s", label="measured (AME2020)")
w = (pr["B_hi0.95"] - pr["B_lo0.95"]).clip(upper=np.nanpercentile(pr["B_hi0.95"] - pr["B_lo0.95"], 98))
sc = ax.scatter(pr.N, pr.Z, s=3, c=w, cmap="viridis", marker="s")
plt.colorbar(sc, label="width of 95% mass interval (MeV)")
ax.scatter(pr[~pr.bound_95].N, pr[~pr.bound_95].Z, s=1, c="#e8590c", marker=".", label="bound status uncertain (95%)")
ax.set_xlabel("N"); ax.set_ylabel("Z"); ax.legend(loc="upper left", markerscale=4, fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/figures/interval_width_chart.pdf"); plt.show()

fig, ax = plt.subplots(figsize=(5, 3.6))
for lv in final.levels:
    ax.plot(range(1, K_PEEL + 1), final.c[lv], marker="o", label=f"level {lv:.3g}")
ax.set_xlabel("distance from core d"); ax.set_ylabel("conformal constant $c_d$ (MeV)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/figures/peeling_curves.pdf"); plt.show()

if len(backtest):
    fig, ax = plt.subplots(figsize=(7, 3.6))
    q = backtest.reset_index(drop=True)
    for j, lv in enumerate(ALPHA_LEVELS):
        s = q[q.level == lv]; x = np.arange(len(s)) + (j - 0.5) * 0.25
        ax.errorbar(x, s.coverage, yerr=[s.coverage - s.CP_lo, s.CP_hi - s.coverage], fmt="o", capsize=3, label=f"nominal {lv}")
        ax.axhline(lv, ls="--", lw=0.8, color=f"C{j}")
    ax.set_xticks(range(len(s))); ax.set_xticklabels(s.quantity, rotation=0); ax.set_ylabel("back-test coverage")
    ax.set_ylim(0.4, 1.02); ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/figures/backtest_coverage.pdf"); plt.show()

## 10. Explanation of Tables (auto-generated)

The text below is written with the numbers of this run and saved as Markdown and LaTeX. Edit it as needed for the journal.

In [ ]:
def fmt(x, n=3):
    return "n/a" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{x:.{n}f}"


n_exp = int((T.status == "exp").sum()); n_pred = int((T.status == "pred").sum())
bt_lines = []
for _, r in backtest.iterrows():
    bt_lines.append(f"| {r.quantity} | {r.level:.2f} | {r.n} | {r.coverage:.3f} | [{r.CP_lo:.3f}, {r.CP_hi:.3f}] | {r.mean_width_MeV:.2f} |")
cal_lines = [f"| {k} | {final.n_d[k-1]} | " + " | ".join(f"{final.c[lv][k-1]:.3f}" for lv in final.levels) + " |"
             for k in range(1, K_PEEL + 1)]

EXPL = f'''# Explanation of Tables

Generated {datetime.date.today().isoformat()}. Base model: **{BASE_NAME}**. Learner: gradient-boosted quantile regression.
Calibration: Peel-CQR with K = {K_PEEL} peeled layers. Fitting set: {len(EXP)} nuclei with Z, N >= {MIN_Z} measured in AME2020.

The table lists {len(T)} nuclei with {MIN_Z} <= Z <= {Z_MAX}: {n_exp} measured in AME2020 and {n_pred} predicted.

## Columns

| Column | Unit | Meaning |
|---|---|---|
| Z, N, A, El | | proton number, neutron number, mass number, element symbol |
| status | | `exp` = mass measured in AME2020; `pred` = not measured |
| d | | extrapolation distance: Manhattan distance |dZ|+|dN| to the nearest measured nucleus (0 for measured) |
| beyond_calibration | | True when d > K = {K_PEEL}: interval uses the linear extension of the calibration curve and carries no coverage guarantee |
| beyond_training_Z | | True when Z exceeds the largest measured Z |
| bound_95 | | True when the lower 95% limits of S_n, S_2n, S_p and S_2p are all positive |
| B_exp, dB_exp, ME_exp | MeV | AME2020 experimental binding energy, its uncertainty and mass excess (measured nuclei) |
| B | MeV | model binding energy (median prediction) |
| B_lo0.68, B_hi0.68 | MeV | 68% prediction interval for B |
| B_lo0.95, B_hi0.95 | MeV | 95% prediction interval for B |
| ME, ME_lo*, ME_hi* | MeV | model atomic mass excess and its intervals, ME = Z*Delta_H + N*Delta_n - B |
| B_AME_syst | MeV | AME2020 value estimated from systematics (#), for comparison only |
| Sn, S2n, Sp, S2p | MeV | one- and two-neutron and -proton separation energies |
| Qa, Qbm | MeV | alpha-decay and beta-minus-decay energies |
| X_lo0.68 ... X_hi0.95 | MeV | 68% and 95% intervals for each derived quantity X |

Constants: Delta_H = {DELTA_H*1000:.6f} keV, Delta_n = {DELTA_N*1000:.6f} keV, B(4He) = {B_ALPHA*1000:.3f} keV (AME2020).

## Definition of the uncertainties

**Masses.** The residual y = B_exp - B_theory is modelled by quantile regression at levels (1-L)/2 and (1+L)/2 for every reported level L.
The outer {K_PEEL} layers of the measured chart are removed ("peeled"); the quantile model is refitted on the remaining core
({final.n_core} nuclei), and the peeled nuclei ({final.n_peel}) sit at distances d = 1..{K_PEEL} from the core. For each level L and distance d,
the conformal constant c_L(d) is the ceil((n_d+1)L)-th smallest conformity score max(q_lo - y, y - q_hi) among the peeled nuclei at distance d.
The final quantile model is fitted on all measured nuclei, and a nucleus at distance d receives [q_lo - c_L(d), q_hi + c_L(d)].
Under the assumption that the score of a future nucleus at distance d is exchangeable with the peeled scores at the same distance,
the interval covers the true binding energy with probability at least L (split-conformal argument), separately for each d <= {K_PEEL}.
For d > {K_PEEL}, c_L(d) is extended linearly (slope fitted on d = 1..{K_PEEL}) and the ML correction is damped as exp(-(d-{K_PEEL})/{DAMP_LENGTH}); these rows are flagged.

| d | peeled nuclei | ''' + " | ".join(f"c_{lv:.3g} (MeV)" for lv in final.levels) + ''' |
|---|---|''' + "---|" * len(final.levels) + "\n" + "\n".join(cal_lines) + f'''

**Derived quantities.** Each derived quantity is a difference of two binding energies plus a constant. Its interval at level L is
[lo_X - hi_Y, hi_X - lo_Y] built from mass intervals at level 1 - (1-L)/2; by the union bound its coverage is at least L for any dependence
between the two mass errors. These intervals are therefore conservative. Measured neighbours enter with their AME2020 value.

## Validation (back-test)

The full procedure was re-run using only nuclei measured in AME2012 and scored on the nuclei first measured in AME2016 or AME2020.
Coverage with 95% Clopper-Pearson intervals:

| Quantity | Nominal | n | Coverage | CP 95% | Mean width (MeV) |
|---|---|---|---|---|---|
''' + "\n".join(bt_lines) + '''

## Caveats

Coverage statements are marginal (averaged over nuclei at a given distance), not conditional on a particular nucleus or region.
Nuclei flagged beyond_calibration or beyond_training_Z rest on extrapolation of the calibration curve and of the base model.
'''
open(f"{OUT_DIR}/explanation_of_tables.md", "w", encoding="utf-8").write(EXPL)
print(EXPL[:3000])

## 11. Write the table files

In [ ]:
# ---- CSV and Excel
T.to_csv(f"{OUT_DIR}/mass_table_full.csv", index=False, float_format="%.4f")
with pd.ExcelWriter(f"{OUT_DIR}/mass_table.xlsx", engine="openpyxl") as xw:
    T.round(4).to_excel(xw, sheet_name="table", index=False)
    pd.DataFrame({"text": EXPL.split("\n")}).to_excel(xw, sheet_name="explanation", index=False)
    if len(backtest):
        backtest.round(4).to_excel(xw, sheet_name="validation", index=False)
        backtest_d.round(4).to_excel(xw, sheet_name="validation_by_distance", index=False)
    calib.round(4).reset_index().to_excel(xw, sheet_name="calibration", index=False)
    for ws in xw.book.worksheets:
        ws.freeze_panes = "A2"

# ---- fixed-width ASCII (keV, AME-like)
hdr = [f"Peel-CQR mass table. Base model: {BASE_NAME}. Generated {datetime.date.today().isoformat()}.",
       "Energies in keV. Intervals: [lo68, hi68] and [lo95, hi95]. Flags: e=measured (AME2020), p=predicted,",
       f"  *=beyond calibrated distance (d>{K_PEEL}), ?=bound status uncertain at 95%.",
       "", f"{'N':>4}{'Z':>4}{'A':>4} {'El':<3}{'f':>2}{'d':>3}{'ME':>12}{'ME_lo68':>12}{'ME_hi68':>12}{'ME_lo95':>12}"
       f"{'ME_hi95':>12}{'Sn':>10}{'S2n':>10}{'Sp':>10}{'S2p':>10}{'Qa':>10}{'Qbm':>10}", "-" * 154]
lines = []
for _, r in T.iterrows():
    f = ("e" if r.status == "exp" else "p") + ("*" if r.beyond_calibration else ("?" if not r.bound_95 else " "))
    if r.status == "exp":                      # measured: experimental value, no model interval
        vals = [r.ME_exp] + [np.nan] * 4
    else:
        vals = [r.ME, r["ME_lo0.68"], r["ME_hi0.68"], r["ME_lo0.95"], r["ME_hi0.95"]]
    der_v = [r[s] for s in DERIVED]
    s = f"{r.N:4d}{r.Z:4d}{r.A:4d} {r.El:<3}{f:>2}{int(r.d):3d}" + "".join(f"{v*1000:12.1f}" if np.isfinite(v) else f"{'':>12}" for v in vals) + \
        "".join(f"{v*1000:10.1f}" if np.isfinite(v) else f"{'':>10}" for v in der_v)
    lines.append(s)
open(f"{OUT_DIR}/mass_table_ascii.txt", "w", encoding="utf-8").write("\n".join(hdr + lines) + "\n")

# ---- LaTeX longtable (printable): ME with asymmetric 68% errors, separation and Q values with 68% errors
def pm(v, lo, hi, scale=1.0, nd=2):
    if not np.isfinite(v):
        return ""
    if abs(hi - lo) < 1e-9:                     # both masses measured: experimental value
        return f"${v*scale:.{nd}f}$"
    return f"${v*scale:.{nd}f}^{{+{(hi-v)*scale:.{nd}f}}}_{{-{(v-lo)*scale:.{nd}f}}}$"


tex = [r"% Requires \usepackage{longtable,booktabs}",
       r"\begin{longtable}{rrrlcr" + "c" * 7 + "}",
       r"\caption{Peel-CQR mass table: mass excess (MeV) and derived energies (MeV) with 68\% intervals. "
       r"Flags: e measured, p predicted, $*$ beyond calibrated distance, ? bound status uncertain at 95\%.}\\",
       r"\toprule", r"$Z$ & $N$ & $A$ & El & f & $d$ & ME & $S_n$ & $S_{2n}$ & $S_p$ & $S_{2p}$ & $Q_\alpha$ & $Q_{\beta^-}$ \\",
       r"\midrule\endfirsthead", r"\toprule $Z$ & $N$ & $A$ & El & f & $d$ & ME & $S_n$ & $S_{2n}$ & $S_p$ & $S_{2p}$ & $Q_\alpha$ & $Q_{\beta^-}$ \\ \midrule\endhead",
       r"\bottomrule\endfoot"]
for _, r in T.iterrows():
    f = ("e" if r.status == "exp" else "p") + ("$*$" if r.beyond_calibration else ("?" if not r.bound_95 else ""))
    me = f"${r.ME_exp:.4f}$" if r.status == "exp" else pm(r.ME, r["ME_lo0.68"], r["ME_hi0.68"])
    ds = [pm(r[s], r[f"{s}_lo0.68"], r[f"{s}_hi0.68"]) for s in DERIVED]
    tex.append(f"{r.Z} & {r.N} & {r.A} & {r.El} & {f} & {int(r.d)} & {me} & " + " & ".join(ds) + r" \\")
tex.append(r"\end{longtable}")
open(f"{OUT_DIR}/mass_table_longtable.tex", "w", encoding="utf-8").write("\n".join(tex) + "\n")

# ---- explanation in LaTeX (simple conversion of the markdown)
import re
tx = EXPL.replace("_", r"\_").replace("%", r"\%").replace("#", r"\#")
tx = re.sub(r"\*\*(.+?)\*\*", r"\\textbf{\1}", tx)
open(f"{OUT_DIR}/explanation_of_tables.tex", "w", encoding="utf-8").write("% Draft: convert the Markdown tables to tabular before submission\n" + tx)

print("files:", sorted(os.listdir(OUT_DIR)), sorted(os.listdir(f"{OUT_DIR}/figures")))

In [ ]:
# ---- quick look
w68 = T["B_hi0.68"] - T["B_lo0.68"]; w95 = T["B_hi0.95"] - T["B_lo0.95"]
summary = (T.assign(dbin=np.where(T.d >= 6, "6+", T.d.astype(int).astype(str)), w68=w68, w95=w95)
             .groupby("dbin").agg(n=("Z", "size"), mean_w68_MeV=("w68", "mean"), mean_w95_MeV=("w95", "mean"),
                                  bound95_share=("bound_95", "mean")))
summary.round(3)

In [ ]:
# ---- zip and download
zpath = "adndt_mass_table_output.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(OUT_DIR):
        for f in fs:
            z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), OUT_DIR))
print("saved", zpath, f"{os.path.getsize(zpath)/1e6:.1f} MB")
if IN_COLAB:
    colab_files.download(zpath)

## Before submitting

1. Set `BASE_FILE` to a strong base model (WS4, FRDM2012 or HFB-24) and rerun. The liquid-drop base is for testing only.
2. Check the back-test table: every nominal level should lie inside its Clopper–Pearson interval.
3. Decide with the editor how far beyond the calibrated distance to print the table (`beyond_calibration` rows), and whether to stop at $Z_{\max}$ lower than 118.
4. Convert the Markdown tables in `explanation_of_tables.tex` to `tabular` and merge it with the journal's *Explanation of Tables* template.